In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_movies")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
def categorizar_votos(cantidad_votos):
    """
    Categorizes a film based on the number of votes received.
    """
    if cantidad_votos >= 100000:
        return "Blockbuster / Mass-market phenomenon"
    elif cantidad_votos >= 10000:
        return "Popular / Commercial success"
    elif cantidad_votos >= 1001:
        return "Moderately well-known"
    elif cantidad_votos >= 100:
        return "Independent / Niche film"
    elif cantidad_votos > 0:
        return "Very obscure / No audience"
    else:
        return "0 votes"

In [0]:
categorizar_votos_uds = F.udf(categorizar_votos, StringType())

In [0]:
df_movie = spark.table(f"{catalogo}.{esquema_source}.movies")
df_moviedetails = spark.table(f"{catalogo}.{esquema_source}.moviedetails")

In [0]:
df_movie = df_movie.dropna(how="all").filter(col("movie_id").isNotNull())

df_moviedetails = df_moviedetails.dropna(how="all").filter(col("movie_id").isNotNull()) 

In [0]:
df_movie = df_movie.withColumn("vote_category", categorizar_votos_uds("vote_count")) \
                    .withColumn("release_year", year(col("release_date"))) \
                    .withColumn("title", trim(col("title"))) \
                    .withColumn("genres", trim(col("genres"))) \
                    .withColumn( "total_minutos", (col("runtime_hour") * 60) + col("runtime_min")
)


df_moviedetails = df_moviedetails.withColumn( 
                                            "budget", 
                                            coalesce(
                                                col("budget_usd").cast(DecimalType(36, 2))
                                                ,lit(0).cast(DecimalType(10, 2)))
                                            ) \
                                            .withColumn( 
                                            "revenue", 
                                            coalesce(
                                                col("revenue_usd").cast(DecimalType(36, 2))
                                                , lit(0).cast(DecimalType(10, 2)))
                                            ) 

df_moviedetails = df_moviedetails.withColumn("profit", (col("revenue") - col("budget")).cast(DecimalType(36, 2)))

In [0]:
df_joined = df_movie.alias("a").join(df_moviedetails.alias("b"), col("a.movie_id") == col("b.movie_id"), "left")

In [0]:
df_filtered_sorted = df_joined.orderBy("budget", ascending=False)

In [0]:
df_filtered_sorted = df_filtered_sorted.withColumn("status_ticket", 
                when(col("budget").isNull() | (col("budget") == 0), lit(None).cast("string"))
                .when(col("revenue") >= (col("budget") * 2.5), "Commercial Success 🔥")
                .when(col("revenue") < col("budget"), "Failure / Loss 📉")
                .otherwise("Break-even Point ⚖️")) 

In [0]:
df_updated = df_filtered_sorted.select("*") \
            .drop(col("a.runtime_min"), col("a.runtime_hour"), col("a.ingestion_date"),
                  col("b.ingestion_date"), col("b.revenue_usd"), col("b.budget_usd"), col("b.movie_id")) \
            .withColumn("slv_date", current_timestamp())

In [0]:
# display(df_updated.limit(10))

In [0]:
df_updated.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.movies_transformed")

In [0]:
%sql
select count(1) from catalog_movies.silver.movies_transformed